# building user define tokenizer using **Transformers**

In [14]:
import transformers
from datasets import load_dataset 

dataset = load_dataset("wikitext",name="wikitext-2-raw-v1",split="train")

def get_training_corpus():
    for i in range(0,len(dataset),1000):
        yield dataset[i:i+1000]["text"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

**Building Wordpiece Tokenizer from scratch**

In [ ]:
from tokenizers import (decoders,models,normalizers,pre_tokenizers,processors,trainers,Tokenizer)
tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))

# Step-1 Normalization
tokenizer.normalizer = normalizers.Sequence(
    [normalizers.NFD(),normalizers.Lowercase(),normalizers.StripAccents()]
)

# Step-2 Pre-tokenization 
tokenizer.pre_tokenizer = pre_tokenizers.Sequence(
    [pre_tokenizers.Whitespace(),pre_tokenizers.Punctuation()]
)

# Step-3 Tokenizer model training 
special_tokens = ["[UNK]","[PAD]","[CLS]","[SEP]","[MASK]"]
trainer = trainers.WordPieceTrainer(vocab_size=25000,special_tokens=special_tokens)
tokenizer.train_from_iterator(get_training_corpus(),trainer=trainer)

# Step-4 post-processing 
cls_token_id = tokenizer.token_to_id("[CLS]")
sep_token_id = tokenizer.token_to_id("[SEP]")
tokenizer.post_processor = processors.TemplateProcessing(
    single=f"[CLS]:0 $A:0 [SEP]:1",
    pair=f"[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1",
    special_tokens = [("[CLS]",cls_token_id),("[SEP]",sep_token_id)]
)

# Creating decoder for decoding encoded input sequences 
tokenizer.decoder = decoders.WordPiece(prefix="##")

In [ ]:
# test tokenizer 
encoding = tokenizer.encode("This is test of Tokenizer, let's see how it perform's.")
print(encoding.ids)
decoded_text=  tokenizer.decode(encoding.ids)
print(decoded_text)

[2, 1511, 1390, 3597, 1339, 24300, 18889, 6600, 16, 3005, 11, 61, 2602, 1728, 1391, 1935, 11, 61, 18, 3]
this is test of tok ##eni ##zer , let ' s see how it perform ' s .


**Building Byte-pair Tokenizer from Scratch**

In [28]:
tokenizer = Tokenizer(models.BPE())

# Step-1 pre-tokenization as gpt2 tokenizer does not perform normalization 
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

# Step-2 tokenizer model training 
special_tokens = ["<|endoftext|>"]
trainer = trainers.BpeTrainer(vocab_size=25000,special_tokens=special_tokens)
tokenizer.train_from_iterator(get_training_corpus(),trainer=trainer)

# Step-3 post-processing 
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)

# Creating decoder for decoding the encoded text 
tokenizer.decoder = decoders.ByteLevel()

In [30]:
# testing the tokenizer
encoded_text = tokenizer.encode("This is testing text.")
print(encoded_text.tokens)
decoded_text = tokenizer.decode(encoded_text.ids)
print(decoded_text)

['T', 'h', 'is', 'Ġis', 'Ġtesting', 'Ġtext', '.']
This is testing text.


In [32]:
# converting it into fast tokenizer 
from transformers import PreTrainedTokenizerFast 
wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object = tokenizer,
    eos = "<|endoftext|>",
    bos = "<|endoftext|>"
)
print(wrapped_tokenizer.is_fast)

True


**Building Unigram tokenizer from scratch**

In [40]:
from tokenizers  import Regex
tokenizer = Tokenizer(models.Unigram())

# Step-1 Normalization 
tokenizer.normalizer = normalizers.Sequence(
    [
    normalizers.Replace("``",'"'),
    normalizers.Replace("''",'"'),
    normalizers.NFKD(),
    normalizers.StripAccents(),
    normalizers.Replace(Regex(" {2,}"), " ")
    ]
)

# Step-2 Pre-tokenization 
tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()

# Step-3 tokenizer model training 
special_tokens = ["<sep>","<cls>","<unk>","<pad>","<mask>"]
trainer = trainers.UnigramTrainer(vocab_size=25000,special_tokens=special_tokens,unk_token="<unk>")
tokenizer.train_from_iterator(get_training_corpus(),trainer=trainer)

# Step-4 Post-processing 
cls_token_id = tokenizer.token_to_id("<cls>")
sep_token_id = tokenizer.token_to_id("<sep>")

tokenizer.post_processor = processors.TemplateProcessing(
    single="$A:0 <sep>:0 <cls>:2",
    pair="$A:0 <sep>:0 $B:1 <sep>:1 <cls>:2",
    special_tokens=[("<sep>", sep_token_id), ("<cls>", cls_token_id)],
)

# creating decoder for decoding encoded text 
tokenizer.decoder = decoders.Metaspace()

In [42]:
# testing tokenizer 
encoded_text = tokenizer.encode("This is a testing's text please ignore")
print(encoded_text.tokens)
decoded_text = tokenizer.decode(encoded_text.ids)
print(decoded_text)

['▁This', '▁is', '▁', 'a', '▁test', 'ing', "'", 's', '▁text', '▁please', '▁ignore', '<sep>', '<cls>']
This is a testing's text please ignore


In [45]:
from transformers import PreTrainedTokenizerFast

wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    cls_token="<cls>",
    sep_token="<sep>",
    mask_token="<mask>",
    padding_side="left",
)
wrapped_tokenizer.is_fast

True

**Summary :**\
*Most of tokenizers follow same steps*\
*1. Normalization(normalizers)*\
*2. Pre-tokenization(pre_tokenizer)*\
*3. Model training(trainers)*\
*4. Post processing(processor)*